In [1]:
# ---------------------------------------------------------------------
# LOCALIZAÇÃO ORIGINAL DOS ARQUIVOS NO GOOGLE DRIVE:
# https://drive.google.com/drive/folders/1OvtUfbhoOjppGK9T_pmH7bYsGbj2Tu9i
# ---------------------------------------------------------------------

import zipfile
from pathlib import Path

# Caminho do arquivo zip enviado/baixado
zip_path = Path("/content/drive-download-20260812T171922Z-1-001.zip")
extract_to = Path("/content/documents_extracted")

# Cria o diretório de extração se não existir
extract_to.mkdir(parents=True, exist_ok=True)

# Descompacta o arquivo
if zip_path.exists():
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Arquivos extraídos com sucesso em: {extract_to}")
else:
    print(f"Arquivo zip não encontrado no caminho: {zip_path}. Verifique o diretório.")

# Lista os arquivos PDF encontrados na extração
pdf_files = list(extract_to.rglob("*.pdf"))
print(f"Total de PDFs encontrados: {len(pdf_files)}")
for pdf in pdf_files:
    print(f" - {pdf.name}")

Arquivos extraídos com sucesso em: /content/documents_extracted
Total de PDFs encontrados: 12
 - llama_foundation_models.pdf
 - attention_is_all_you_need.pdf
 - gpt4_technical_report.pdf
 - twitter_algoritmo.pdf
 - retrieval_augmented_generation.pdf
 - gpt3_language_models.pdf
 - scaling_laws_llm.pdf
 - lora_low_rank_adaptation.pdf
 - escrita_academica_ia.pdf
 - bert_pretraining.pdf
 - instruct_gpt.pdf
 - bioetica_e_ia.pdf


In [5]:
import os
from pathlib import Path
from langchain_core.documents import Document
import fitz # PyMuPDF

def extrair_pdf_para_markdown(pdf_path: Path, output_md_dir: Path) -> Path:
    output_md_dir.mkdir(parents=True, exist_ok=True)
    md_filename = pdf_path.stem + ".md"
    md_path = output_md_dir / md_filename

    # Extração simples das páginas do PDF para Markdown estruturado
    doc_pdf = fitz.open(str(pdf_path))
    markdown_content = f"# {pdf_path.stem.replace('_', ' ').title()}\n\n"

    for page_num, page in enumerate(doc_pdf):
        text = page.get_text()
        markdown_content += f"## Página {page_num + 1}\n\n{text}\n\n--- \n\n"

    with open(md_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)

    return md_path

# Diretórios de saída
extracted_dir = Path("/content/documents_extracted")
markdown_output_dir = Path("/content/results_markdown")

# Processando todos os PDFs encontrados
pdf_files = list(extracted_dir.rglob("*.pdf"))
print(f"Iniciando a extração de {len(pdf_files)} PDFs para Markdown...")

for pdf_file in pdf_files:
    md_file = extrair_pdf_para_markdown(pdf_file, markdown_output_dir)
    print(f"Convertido: {pdf_file.name} -> {md_file.name}")

print("\nExtração concluída! Os arquivos Markdown estão prontos para receber as 10 estratégias de chunking.")

Iniciando a extração de 12 PDFs para Markdown...
Convertido: llama_foundation_models.pdf -> llama_foundation_models.md
Convertido: attention_is_all_you_need.pdf -> attention_is_all_you_need.md
Convertido: gpt4_technical_report.pdf -> gpt4_technical_report.md
Convertido: twitter_algoritmo.pdf -> twitter_algoritmo.md
Convertido: retrieval_augmented_generation.pdf -> retrieval_augmented_generation.md
Convertido: gpt3_language_models.pdf -> gpt3_language_models.md
Convertido: scaling_laws_llm.pdf -> scaling_laws_llm.md
Convertido: lora_low_rank_adaptation.pdf -> lora_low_rank_adaptation.md
Convertido: escrita_academica_ia.pdf -> escrita_academica_ia.md
Convertido: bert_pretraining.pdf -> bert_pretraining.md
Convertido: instruct_gpt.pdf -> instruct_gpt.md
Convertido: bioetica_e_ia.pdf -> bioetica_e_ia.md

Extração concluída! Os arquivos Markdown estão prontos para receber as 10 estratégias de chunking.


In [8]:
!pip install -q langchain-huggingface sentence-transformers
import os
import json
from pathlib import Path
from typing import List, Dict, Any
from langchain_core.documents import Document
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter
)
from langchain_huggingface import HuggingFaceEmbeddings

class ChunkingExperimentPipeline:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        print(f"Carregando modelo de embeddings: {model_name}...")
        self.embeddings_model = HuggingFaceEmbeddings(model_name=model_name)

    def run_all_tests(self, md_file_path: str, doc_id: str, doc_name: str, output_base_dir: str = "/content/results"):
        path = Path(md_file_path)
        if not path.exists():
            raise FileNotFoundError(f"Arquivo Markdown não encontrado: {md_file_path}")

        with open(path, "r", encoding="utf-8") as f:
            text_content = f.read()

        base_doc = Document(page_content=text_content, metadata={"document_id": doc_id, "document_name": doc_name})

        # Definição dos 10 testes solicitados
        experiments = [
            {"test_id": 1, "strategy": "fixed_200_no_overlap", "type": "fixed", "chunk_size": 200, "chunk_overlap": 0},
            {"test_id": 2, "strategy": "fixed_500_no_overlap", "type": "fixed", "chunk_size": 500, "chunk_overlap": 0},
            {"test_id": 3, "strategy": "fixed_1000_no_overlap", "type": "fixed", "chunk_size": 1000, "chunk_overlap": 0},
            {"test_id": 4, "strategy": "fixed_2000_no_overlap", "type": "fixed", "chunk_size": 2000, "chunk_overlap": 0},
            {"test_id": 5, "strategy": "fixed_500_overlap_50", "type": "fixed", "chunk_size": 500, "chunk_overlap": 50},
            {"test_id": 6, "strategy": "fixed_500_overlap_200", "type": "fixed", "chunk_size": 500, "chunk_overlap": 200},
            {"test_id": 7, "strategy": "by_paragraph", "type": "paragraph"},
            {"test_id": 8, "strategy": "sentences_grouped_3", "type": "sentences"},
            {"test_id": 9, "strategy": "recursive", "type": "recursive"},
            {"test_id": 10, "strategy": "markdown_headers", "type": "markdown"},
        ]

        doc_results_summary = {"document": doc_name, "experiments": []}
        doc_output_dir = Path(output_base_dir) / doc_id

        for exp in experiments:
            chunks = self._apply_splitter(base_doc, exp)
            chunk_data_list = self._process_chunks_and_embeddings(chunks, exp, doc_id, doc_name)

            # Salvar por teste
            test_dir = doc_output_dir / f"test_{exp['test_id']:02d}"
            test_dir.mkdir(parents=True, exist_ok=True)

            output_file = test_dir / "chunks_embeddings.json"
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(chunk_data_list, f, ensure_ascii=False, indent=4)

            # Coleta estatísticas para o summary
            sizes = [len(c["text"]) for c in chunk_data_list]
            avg_size = sum(sizes) / len(sizes) if sizes else 0

            doc_results_summary["experiments"].append({
                "test_id": exp["test_id"],
                "strategy": exp["strategy"],
                "chunk_size": exp.get("chunk_size"),
                "chunk_overlap": exp.get("chunk_overlap", 0),
                "num_chunks": len(chunk_data_list),
                "avg_chunk_size": round(avg_size, 2),
                "embedding_dimension": len(chunk_data_list[0]["embedding"]) if chunk_data_list else 0
            })

        # Salvar summary.json do documento
        summary_path = doc_output_dir / "summary.json"
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(doc_results_summary, f, ensure_ascii=False, indent=4)

    def _apply_splitter(self, doc: Document, exp: Dict[str, Any]) -> List[Document]:
        stype = exp["type"]

        if stype == "fixed":
            splitter = CharacterTextSplitter(separator="", chunk_size=exp["chunk_size"], chunk_overlap=exp["chunk_overlap"])
            return splitter.split_documents([doc])
        elif stype == "paragraph":
            splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=0)
            return splitter.split_documents([doc])
        elif stype == "sentences":
            sentences = [s.strip() for s in doc.page_content.split(".") if s.strip()]
            grouped_docs = []
            group_size = 3
            for i in range(0, len(sentences), group_size):
                group_text = ". ".join(sentences[i:i+group_size]) + "."
                grouped_docs.append(Document(page_content=group_text, metadata=doc.metadata))
            return grouped_docs
        elif stype == "recursive":
            splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", " ", ""], chunk_size=1000, chunk_overlap=200)
            return splitter.split_documents([doc])
        elif stype == "markdown":
            headers_to_split_on = [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")]
            splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
            return splitter.split_text(doc.page_content)
        return [doc]

    def _process_chunks_and_embeddings(self, chunks: List[Document], exp: Dict[str, Any], doc_id: str, doc_name: str) -> List[Dict[str, Any]]:
        texts = [c.page_content for c in chunks]
        embeddings = self.embeddings_model.embed_documents(texts) if texts else []

        result = []
        for idx, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
            chunk_id = f"{doc_id}_test{exp['test_id']:02d}_chunk{idx+1:03d}"
            meta = getattr(chunk, "metadata", {})
            chunk_record = {
                "chunk_id": chunk_id,
                "document_id": doc_id,
                "document_name": doc_name,
                "test_id": exp["test_id"],
                "strategy": exp["strategy"],
                "chunk_size": exp.get("chunk_size"),
                "chunk_overlap": exp.get("chunk_overlap", 0),
                "text": chunk.page_content,
                "embedding": embedding,
                "metadata": meta
            }
            result.append(chunk_record)
        return result

# Inicializando o pipeline e processando os Markdowns gerados
pipeline = ChunkingExperimentPipeline()
markdown_dir = Path("/content/results_markdown")
md_files = list(markdown_dir.glob("*.md"))

print(f"\nIniciando experimentos de chunking e embeddings para {len(md_files)} documentos...")
for idx, md_file in enumerate(md_files):
    doc_id = f"doc{idx+1:02d}"
    print(f"Processando [{idx+1}/12]: {md_file.name}")
    pipeline.run_all_tests(str(md_file), doc_id, md_file.name)

print("\nTudo pronto! Todos os 10 testes de chunking e embeddings foram gerados e salvos em /content/results.")

Carregando modelo de embeddings: sentence-transformers/all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Processando [9/12]: instruct_gpt.md


Processando [10/12]: gpt3_language_models.md


Processando [11/12]: attention_is_all_you_need.md


Processando [12/12]: twitter_algoritmo.md



Tudo pronto! Todos os 10 testes de chunking e embeddings foram gerados e salvos em /content/results.
